# Cleaning

In [ ]:
# Import, clean

import pandas as pd

games = pd.read_csv("data/games.csv")
matches = pd.read_csv("data/matches.csv")
tournaments = pd.read_csv("data/tour_meta.csv")

# Clean
# drop missing dates (games)
games['date'] = pd.to_datetime(games.date.str.split('(').str[0].str.strip())
games = games[pd.notna(games['date'])]
# sort by date
games.sort_values('date', inplace=True)

# clean dates (matches)
matches['date'] = pd.to_datetime(matches.date)
matches = matches[pd.notna(matches['date'])]
# sort by date
matches.sort_values('date', inplace=True)
# drop nan scores
matches = matches[pd.notna(matches['team1_score']) & pd.notna(matches['team2_score'])]

In [ ]:
# Tournament attributes

import re
import pandas as pd

def map_tournament_attributes(matches: pd.DataFrame) -> pd.DataFrame:

    def get_region(name: str) -> str:
        if name.startswith("LEC"):
            return "LEC"
        if name.startswith(("LCK", "LCK Cup")):
            return "LCK"
        if name.startswith(("LPL", "LPL ")):
            return "LPL"
        if name.startswith(("LCS", "LTA", "LCP")):
            return "LCS"
        if name.startswith(("MSI", "Mid-Season Invitational")):
            return "INTL"
        if "World" in name or "Esports World Cup" in name:
            return "INTL"
        return "OTHER"

    def get_season(name: str) -> int | None:
        m = re.search(r"(20\d{2})", name)  # pl 2023/2024/2025
        return int(m.group(1)) if m else None

    def get_split(name: str) -> str:
        # Normalize split: Spring/Winter → Split1, Summer/Split2 → Split2, MSI/Worlds/EWC → International
        lowered = name.lower()
        if any(x in lowered for x in ["msi", "world", "esports world cup"]):
            return "International"
        if any(x in lowered for x in ["winter", "spring", "split 1"]):
            return "Split1"
        if any(x in lowered for x in ["summer", "split 2"]):
            return "Split2"
        return "Other"

    def get_stage(name: str) -> str:
        lowered = name.lower()
        if "groups" in lowered:
            return "Groups"
        if "playoffs" in lowered:
            return "Playoffs"
        if "regional finals" in lowered:
            return "Regional Finals"
        if "championship" in lowered:
            return "Championship"
        if "play-in" in lowered or "qualifying" in lowered:
            return "Qualifier"
        if "main event" in lowered:
            return "Main Event"
        # alap esetben Season
        if any(x in lowered for x in ["season", "split", "rounds"]):
            return "Season"
        return "Unknown"

    def is_international(name: str) -> int:
        lowered = name.lower()
        return 1 if any(x in lowered for x in ["msi", "world", "esports world cup"]) else 0

    def is_regional_final(name: str) -> int:
        return 1 if "regional finals" in name.lower() else 0

    matches["region"] = matches["tournament_name"].apply(get_region)
    matches["season"] = matches["tournament_name"].apply(get_season)
    matches["split"] = matches["tournament_name"].apply(get_split)
    matches["stage"] = matches["tournament_name"].apply(get_stage)
    matches["is_international"] = matches["tournament_name"].apply(is_international)
    matches["is_regional_final"] = matches["tournament_name"].apply(is_regional_final)

    return matches


tournaments = map_tournament_attributes(tournaments)
display(tournaments.sample(5))
print(tournaments[['tournament_name', 'region', 'season', 'split', 'stage', 'is_international', 'is_regional_final']].info())

In [ ]:
# ELO calc

import pandas as pd

matches = pd.merge(matches, tournaments[['tournament_name', 'region', 'is_international']], 
                   how='left', on='tournament_name')

# kezdő ELO minden csapatnak
initial_elo = 1500
# példa: ligánként eltérő kezdő ELO
league_base_elo = {
    "LPL": 1550,
    "LCK": 1550,
    "LEC": 1525,
    "LCS": 1500,
    "LTA": 1450,
    "LCP": 1450,
    "MSI": 1580,  # ha nemzetközi bónusz
    "Worlds": 1600
}
K = 20  # standard K érték, meccsenként

# létrehozunk egy dictionary-t a csapatok ELO-jához
team_elos = {}

# új oszlopok a matches-hez
matches['team1_elo_before'] = 0
matches['team2_elo_before'] = 0
matches['team1_elo_after'] = 0
matches['team2_elo_after'] = 0

# Liga matchup korrekció: international meccsekből számoljuk
international_matches = matches[matches.get('is_international', 0) == 1]

# Liga matchup dict létrehozása
league_correction = {}  # pl. ('LPL','LEC') -> +20
for idx, row in international_matches.iterrows():
    l1, l2 = row['region'], row['region']  # ha külön kell: team1_region, team2_region
    l1, l2 = row['team1_region'] if 'team1_region' in row else l1, row['team2_region'] if 'team2_region' in row else l2
    key = (l1, l2)
    league_correction.setdefault(key, []).append(1 if row['team1_score'] > row['team2_score'] else 0)

# Átlagos korrekció számítása
for k in league_correction:
    # winrate: 0-1 → átalakítjuk Elo pontban, pl. *100
    league_correction[k] = (sum(league_correction[k]) / len(league_correction[k]) - 0.5) * 200  

# rendezzük időrendbe
matches = matches.sort_values('date').reset_index(drop=True)

for idx, row in matches.iterrows():
    t1 = row['team1']
    t2 = row['team2']
    league1 = row['region']
    league2 = row['region']
    
    if t1 not in team_elos:
        base = league_base_elo.get(league1, initial_elo)
        team_elos[t1] = base
    if t2 not in team_elos:
        base = league_base_elo.get(league2, initial_elo)
        team_elos[t2] = base

    elo1 = team_elos[t1]
    elo2 = team_elos[t2]

    matches.at[idx, 'team1_elo_before'] = elo1
    matches.at[idx, 'team2_elo_before'] = elo2

    # Liga matchup korrekció hozzáadása
    correction = league_correction.get((league1, league2), 0)
    
    expected1 = 1 / (1 + 10 ** (((elo2 - elo1) - correction) / 400))
    expected2 = 1 / (1 + 10 ** (((elo1 - elo2) + correction) / 400))

    if row['team1_score'] > row['team2_score']:
        s1, s2 = 1, 0
    else:
        s1, s2 = 0, 1

    elo1_new = elo1 + K * (s1 - expected1)
    elo2_new = elo2 + K * (s2 - expected2)

    team_elos[t1] = elo1_new
    team_elos[t2] = elo2_new

    matches.at[idx, 'team1_elo_after'] = elo1_new
    matches.at[idx, 'team2_elo_after'] = elo2_new

team_elos

In [ ]:
# ELO visualization

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------
# 1️⃣ ELO rangsor jelenleg
# -------------------------------
# Kiszámítjuk a végső ELO-t minden csapatra (utolsó meccs után)
elo_latest_team1 = matches.groupby('team1')['team1_elo_after'].last()
elo_latest_team2 = matches.groupby('team2')['team2_elo_after'].last()

elo_latest = pd.concat([elo_latest_team1, elo_latest_team2], axis=1)
elo_latest['final_elo'] = elo_latest.max(axis=1)
elo_rank = elo_latest['final_elo'].sort_values(ascending=False)
print("=== Jelenlegi ELO rangsor ===")
print(elo_rank.head(10))

# -------------------------------
# 2️⃣ Upset / meglepetés meccsek
# -------------------------------
# Definiáljuk, hogy upset, ha a győztes ELO-ja legalább 100 ponttal kevesebb volt a vesztesnél
def detect_upset(row):
    if row['team1_score'] > row['team2_score'] and row['team1_elo_after'] + 100 < row['team2_elo_after']:
        return True
    elif row['team2_score'] > row['team1_score'] and row['team2_elo_after'] + 100 < row['team1_elo_after']:
        return True
    else:
        return False

matches['upset'] = matches.apply(detect_upset, axis=1)
num_upsets = matches['upset'].sum()
print(f"Number of upsets: {num_upsets}")

# -------------------------------
# 3️⃣ Ligánkénti ELO trend
# -------------------------------
plt.figure(figsize=(14,6))
sns.lineplot(data=matches, x='date', y='team1_elo_after', hue='team1', alpha=0.5)
plt.title("ELO trend csapatonként ligánként (team1 ELO)")
plt.xlabel("Date")
plt.ylabel("ELO")
plt.legend([],[], frameon=False)  # túl sok csapat, ezért elrejtjük a legend-et
plt.show()

# -------------------------------
# 4️⃣ ELO boxplot csapatonként
# -------------------------------
plt.figure(figsize=(16,6))
top_teams = matches['team1'].value_counts().nlargest(10).index  # Top 10 gyakori csapat
sns.boxplot(data=matches[matches['team1'].isin(top_teams)], x='team1', y='team1_elo_after')
plt.title("ELO boxplot csapatonként (top 10 gyakori csapat)")
plt.xlabel("Team")
plt.ylabel("ELO")
plt.xticks(rotation=45)
plt.show()

# Odds

## OddsPortal

In [ ]:
# Tournaments

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

def scrape_oddsportal_lol_links():
    url = "https://www.oddsportal.com/results/#esports"
    driver = webdriver.Chrome()
    driver.get(url)

    wait = WebDriverWait(driver, 15)
    # várjuk, amíg betöltődik legalább 1 tournament link
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li[data-testid='results-tournament-box'] a")))

    time.sleep(2)  # kis extra wait, hogy minden JS lefusson

    links = driver.find_elements(By.CSS_SELECTOR, "li[data-testid='results-tournament-box'] a")

    data = []
    for link in links:
        try:
            text = link.text.strip()
            href = link.get_attribute("href")
            if text.startswith("League of Legends "):
                name = text.replace("League of Legends ", "").strip()
                data.append({
                    "Name": name,
                    "url": href
                })
        except Exception as e:
            print(f"⚠️ Hiba egy link feldolgozásánál: {e}")
            continue

    driver.quit()
    df = pd.DataFrame(data)
    print(f"✅ {len(df)} League of Legends esemény található az OddsPortalon.")
    return df


# --- Példa futtatás ---
df_oddsportal = scrape_oddsportal_lol_links()
display(df_oddsportal)


In [ ]:
# Odds of tournament

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import re, time
from datetime import datetime

def scrape_oddsportal_fixed(event_url, headless=False):
    opts = webdriver.ChromeOptions()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("user-agent=Mozilla/5.0")
    
    driver = webdriver.Chrome(options=opts)
    driver.get(event_url)
    wait = WebDriverWait(driver, 20)

    # --- Cookie gomb elfogadása, ha van ---
    try:
        cookie_btn = wait.until(EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler")))
        cookie_btn.click()
        time.sleep(1)
        print("🍪 Cookie elfogadva")
    except:
        pass

    # --- 2023 és 2024 szezon linkek lekérdezése ---
    season_links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/results/']")
    season_urls = []
    season_urls.append(event_url)
    for a in season_links:
        href = a.get_attribute("href")
        if re.search(r'2023/results/', href) or re.search(r'2024/results/', href):
            season_urls.append(href)

    # Ha nincs külön link, használjuk az eredeti event_url-t
    if not season_urls:
        season_urls = [event_url]

    all_data = []

    # --- Iterálunk a szezon linkeken ---
    for season_url in season_urls:
        driver.get(season_url)
        time.sleep(2)
        print(f"📅 Szezon feldolgozása: {season_url}")

        page = 1
        while True:
            print(f"🔍 Oldal {page} feldolgozása...")
            
            # Inkrementális scroll: lépésekben, lassabban
            scroll_pause = 1.5
            scroll_step = 500  # pixelenként
            last_height = driver.execute_script("return document.body.scrollHeight")
            current_pos = 0

            while current_pos < last_height:
                driver.execute_script(f"window.scrollTo(0, {current_pos});")
                time.sleep(scroll_pause)
                current_pos += scroll_step
                new_height = driver.execute_script("return document.body.scrollHeight")
                if new_height > last_height:
                    last_height = new_height


            time.sleep(2)
            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.eventRow")))

            event_rows = driver.find_elements(By.CSS_SELECTOR, "div.eventRow")
            current_date = None

            for event in event_rows:
                # dátum keresése
                date_found = False
                
                # Dátum keresése az event teljes szövegében
                date_text = event.text.strip()
                if any(month in date_text for month in ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']):
                    lines = date_text.split('\n')
                    for line in lines:
                        if any(month in line for month in ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']):
                            # Dátum formázása - eltávolítjuk a " -" utáni részt
                            clean_date = line.split(' -')[0].strip()
                            current_date = clean_date
                            
                            # Dátum átalakítása YYYY-MM-dd formátumba
                            try:
                                date_obj = datetime.strptime(current_date, '%d %b %Y')
                                current_date = date_obj.strftime('%Y-%m-%d')
                            except ValueError:
                                # Ha nem sikerül átalakítani, marad az eredeti
                                pass
                            
                            date_found = True
                            break

                # Ha nincs dátum, de van game-row, akkor meccs sor
                try:
                    game_row = event.find_element(By.CSS_SELECTOR, "div[data-testid='game-row']")
                except:
                    continue

                # csapatnevek
                participants = game_row.find_elements(By.CSS_SELECTOR, "a[title]")
                if len(participants) < 2:
                    continue

                home_team = participants[0].get_attribute("title").strip()
                away_team = participants[1].get_attribute("title").strip()

                # oddsok
                odds_blocks = event.find_elements(By.CSS_SELECTOR, "div[data-testid^='odd-container'] p")
                odds = []
                for p in odds_blocks:
                    try:
                        odds_text = p.text.strip().replace(",", ".")
                        if re.match(r"^\d+(\.\d+)?$", odds_text):
                            odds.append(float(odds_text))
                    except:
                        continue
                
                home_odds = odds[0] if len(odds) > 0 else None
                away_odds = odds[1] if len(odds) > 1 else None

                all_data.append({
                    "Date": current_date,
                    "home_team": home_team,
                    "away_team": away_team,
                    "home_odds": home_odds,
                    "away_odds": away_odds
                })

            # Következő oldal ellenőrzése
            try:
                next_button = driver.find_element(By.CSS_SELECTOR, f"a.pagination-link[data-number='{page + 1}']")
                if next_button.is_enabled():
                    print(f"➡️ Következő oldal: {page + 1}")
                    driver.execute_script("arguments[0].click();", next_button)
                    page += 1
                    time.sleep(3)  # Várakozás az oldal betöltésére
                    continue
                else:
                    break
            except:
                # Ha nincs következő oldal, kilépünk
                break

    driver.quit()
    
    df = pd.DataFrame(all_data).drop_duplicates(subset=["home_team","away_team","home_odds","away_odds"])
    print(f"✅ Összesen {len(df)} meccs feldolgozva {page} oldalról.")
    return df

# Teszt
#url = "https://www.oddsportal.com/esports/league-of-legends/league-of-legends-world-championship/results/"
#df_odds = scrape_oddsportal_fixed(url, headless=False)
#display(df_odds)

In [ ]:
# Relevant odds
  
oddsportal_relevant = [
    "LEC",
    "LCS Lock-In",
    "LTA North",
    "LTA South",
    "LTA Cross Conference",
    "LCP",
    "LCK",
    "LPL",
    "World Championship",
    "Esports World Cup"
]

df_odds_super = pd.DataFrame()
for op_tour in oddsportal_relevant:
    url_op = df_oddsportal[df_oddsportal.Name == op_tour]['url'].iloc[0]
    df_odds = scrape_oddsportal_fixed(url_op, headless=False)
    df_odds['tournament_op'] = op_tour
    df_odds_super = pd.concat([df_odds_super, df_odds], ignore_index=True)

display(df_odds_super)

In [ ]:
# Save odds super

df_odds_super['Date'] = pd.to_datetime(df_odds_super['Date'])
df_odds_super.sort_values('Date', inplace=True)
df_odds_super = df_odds_super.reset_index(drop=True)
display(df_odds_super)

df_odds_super.to_csv('data/odds.csv', index=False)

## Fuzzy match

In [ ]:
# Fuzzy matching JSON

import pandas as pd
from fuzzywuzzy import process
import json
from pathlib import Path
import re

# --- Helper függvény: normalizálás ---
def normalize_team_name(name):
    name = name.lower()
    # eltávolítjuk a 'team', 'gaming', 'esports' szavakat
    name = re.sub(r'\b(team|gaming|esports)\b', '', name)
    # extra whitespace eltávolítás
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# --- Egyedi csapatnevek ---
odds_teams = pd.unique(df_odds_super[['home_team','away_team']].values.ravel())
match_teams = pd.unique(matches[['team1','team2']].values.ravel())

# --- Normalizált változatok készítése fuzzyhoz ---
normalized_match_teams = {normalize_team_name(t): t for t in match_teams}

team_mapping = {}
for team in odds_teams:
    norm_team = normalize_team_name(team)
    
    # fuzzy match a normalizált neveken
    best_match_norm, score = process.extractOne(norm_team, normalized_match_teams.keys())
    # visszakapjuk az eredeti matches csapatnevet
    matched_team = normalized_match_teams[best_match_norm]
    
    team_mapping[team] = {
        "matched_team": matched_team,
        "score": score
    }

# --- Mentés JSON-ba ---
#Path("data").mkdir(exist_ok=True)
#with open("data/team_name_mapping.json", "w", encoding="utf-8") as f:
#    json.dump(team_mapping, f, indent=2, ensure_ascii=False)

#print("✅ Normalizált fuzzy team mapping kész, elmentve: data/team_name_mapping.json")


In [ ]:
# Map team names

with open("data/team_name_mapping.json", "r", encoding="utf-8") as f:
    team_mapping = json.load(f)

for i, row in df_odds_super.iterrows():
    df_odds_super.loc[i, 'team1'] = team_mapping[row['home_team']]['matched_team']
    df_odds_super.loc[i, 'team2'] = team_mapping[row['away_team']]['matched_team']


# Matches mapping
## convert name and type
df_odds_super.rename(columns={'Date': 'date'}, inplace=True)
df_odds_super['date'] = pd.to_datetime(df_odds_super['date'])

## merge manually
for i, row in df_odds_super.iterrows():
    home_op = row['team1']
    away_op = row['team2']

    rel_matches = matches[(matches.date < row['date'] + pd.Timedelta(days=1)) & 
                          (matches.date > row['date'] - pd.Timedelta(days=1))]
    if rel_matches.shape[0] == 0:
        continue
    else:
        for j, match in rel_matches.iterrows():
            if home_op == match['team1'] and away_op == match['team2']:
                matches.loc[j, 'team1_odds'] = row['home_odds']
                matches.loc[j, 'team2_odds'] = row['away_odds']
            elif home_op == match['team2'] and away_op == match['team1']:
                matches.loc[j, 'team1_odds'] = row['away_odds']
                matches.loc[j, 'team2_odds'] = row['home_odds']

display(matches.sample(7))
print(f"Matches with matched odds: {len(matches[matches['team1_odds'].notna()])}/{len(matches)}")